In [2]:
# Imprimir información sobre la división
import pickle
with open("split_data_dict.pkl", "rb") as f:
    obj = pickle.load(f)

prueba = obj["prueba"]
split_data_dict = obj["split"]

for key, data in split_data_dict.items():
    print(f"Dataset: {key}")
    print(f"Tamaño de entrenamiento: {1 - prueba} | prueba: {prueba}")
    print(f"Min/Max de X_train: {data['train']['X'].min()}, {data['train']['X'].max()}")
    print(f"Min/Max de X_test: {data['test']['X'].min()}, {data['test']['X'].max()}")
    print(f"Tamaño -> X_train: {data['train']['X'].shape}, X_test: {data['test']['X'].shape}, "
          f"Y_train: {data['train']['Y'].shape}, Y_test: {data['test']['Y'].shape}\n")

Dataset: orig
Tamaño de entrenamiento: 0.8 | prueba: 0.2
Min/Max de X_train: 0.9745220898716793, 3.9621314639076735
Min/Max de X_test: 0.9748580503748066, 3.726623659510136
Tamaño -> X_train: (5756, 501), X_test: (1440, 501), Y_train: (5756, 1), Y_test: (1440, 1)

Dataset: sg
Tamaño de entrenamiento: 0.8 | prueba: 0.2
Min/Max de X_train: 0.9752886127765829, 3.5910473108689374
Min/Max de X_test: 0.9755120416313424, 3.4683186516323525
Tamaño -> X_train: (5756, 501), X_test: (1440, 501), Y_train: (5756, 1), Y_test: (1440, 1)

Dataset: msc
Tamaño de entrenamiento: 0.8 | prueba: 0.2
Min/Max de X_train: 1.0404674721382379, 3.6052977194138074
Min/Max de X_test: 1.0405483671078637, 3.396062387918411
Tamaño -> X_train: (5756, 501), X_test: (1440, 501), Y_train: (5756, 1), Y_test: (1440, 1)

Dataset: snv
Tamaño de entrenamiento: 0.8 | prueba: 0.2
Min/Max de X_train: -2.811475760461824, 2.2578953916480784
Min/Max de X_test: -2.8084884803188337, 1.846030072642991
Tamaño -> X_train: (5756, 501), X_

In [1]:
import pickle

filename = "best_hyperparams_all_models.pkl"

# 1. Cargar diccionario existente
with open(filename, "rb") as f:
    best_hyperparams = pickle.load(f)

# imprimir
print("\n=== Mejores Hiperparámetros ===")
for model_name, datasets in best_hyperparams.items():
    print(f"\nModelo: {model_name}")
    for dataset, result in datasets.items():
        print(f" Dataset: {dataset}")
        print("  Hiperparámetros:", result["best_params"])
        print(f"  Mejor MSE: {result['best_value']:.5f}")


=== Mejores Hiperparámetros ===

Modelo: random_forest
 Dataset: orig
  Hiperparámetros: {'n_estimators': 430, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
  Mejor MSE: 18.45560
 Dataset: sg
  Hiperparámetros: {'n_estimators': 350, 'max_depth': 45, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
  Mejor MSE: 13.86286
 Dataset: msc
  Hiperparámetros: {'n_estimators': 500, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
  Mejor MSE: 20.70336
 Dataset: snv
  Hiperparámetros: {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
  Mejor MSE: 21.23559

Modelo: svr
 Dataset: orig
  Hiperparámetros: {'C': 29.958171631189153, 'epsilon': 0.04455338175146469, 'kernel': 'rbf', 'gamma': 0.007460250384322396}
  Mejor MSE: 21.35367
 Dataset: sg
  Hiperparámetros: {'C': 28.707297092891597, 'epsilon': 0.4504506686358284, 'kernel': 'rbf', 'ga

### CNN

In [6]:
import os
import time
import random
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# -------------------------
# Configuración general
# -------------------------
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Cuántos trials por dataset
N_TRIALS_PER_DATASET = 25   # recomendado inicial: 20-50 si usar GPU
MAX_EPOCHS = 40             # epochs por trial (puedes bajar a 20 si quieres más trials)
PATIENCE = 6                # early stopping por trial (epochs sin mejora en val)
BATCH_SIZE_CHOICES = [16, 32, 64]

# Workers para DataLoader (usa varios cores para mover datos)
NUM_WORKERS = max(1, (os.cpu_count() or 4) // 4)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -------------------------
# Modelo CNN
# -------------------------
class NIR_CNN(nn.Module):
    def __init__(self, n_features, n_filters=32, kernel_size=5, n_conv_blocks=2, hidden_dim=128, dropout=0.2):
        super().__init__()
        layers = []
        in_ch = 1
        for i in range(n_conv_blocks):
            out_ch = n_filters * (2**i if i>0 else 1)
            layers.append(nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size//2))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(out_ch))
            layers.append(nn.MaxPool1d(kernel_size=2))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = out_ch

        self.conv = nn.Sequential(*layers)

        # calcular tamaño después de las maxpool (cada bloque divide por 2)
        conv_out_len = n_features // (2**n_conv_blocks)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_ch * conv_out_len, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        # x: (batch, n_features)
        x = x.unsqueeze(1)            # -> (batch, 1, n_features)
        x = self.conv(x)              # -> (batch, channels, len)
        x = self.fc(x)
        return x.squeeze(1)

# -------------------------
# Objetivo Optuna (por dataset)
# -------------------------
def make_objective(X_train_orig, y_train_orig, X_val_orig, y_val_orig):
    """
    Retorna una función objective(trial) cerrada sobre los arrays escalados.
    Escala X e y internamente y reporta MSE en unidades originales.
    """
    # escalers locales por dataset
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_orig)
    X_val_scaled   = scaler_X.transform(X_val_orig)

    y_train_scaled = scaler_y.fit_transform(y_train_orig.reshape(-1, 1)).ravel()
    y_val_scaled   = scaler_y.transform(y_val_orig.reshape(-1, 1)).ravel()

    # tensores fijos
    X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32).to(DEVICE)
    y_val_t = torch.tensor(y_val_scaled, dtype=torch.float32).to(DEVICE)

    train_dataset_full = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32),
                                       torch.tensor(y_train_scaled, dtype=torch.float32))

    def objective(trial):
        # hyperparams a buscar
        n_filters = trial.suggest_categorical("n_filters", [16, 32, 64])
        kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7])
        n_conv_blocks = trial.suggest_int("n_conv_blocks", 1, 3)
        hidden = trial.suggest_int("hidden", 64, 256, step=32)
        dropout = trial.suggest_float("dropout", 0.0, 0.4)
        lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        batch_size = trial.suggest_categorical("batch_size", BATCH_SIZE_CHOICES)
        epochs = MAX_EPOCHS

        # crear train/val split interno (recomendado: toma 10-20% de train para val)
        train_ds, val_ds = torch.utils.data.random_split(train_dataset_full, [
            int(len(train_dataset_full) * 0.85),
            len(train_dataset_full) - int(len(train_dataset_full) * 0.85)
        ])

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True)

        model = NIR_CNN(n_features=X_train_orig.shape[1],
                        n_filters=n_filters,
                        kernel_size=kernel_size,
                        n_conv_blocks=n_conv_blocks,
                        hidden_dim=hidden,
                        dropout=dropout).to(DEVICE)

        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        loss_fn = nn.MSELoss()

        best_val = np.inf
        epochs_no_improve = 0

        for epoch in range(1, epochs + 1):
            model.train()
            running_loss = 0.0
            for xb, yb in train_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                optimizer.zero_grad()
                preds = model(xb)
                loss = loss_fn(preds, yb)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * xb.size(0)

            # VALIDACION (rápida, en batch)
            model.eval()
            val_preds = []
            val_targets = []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(DEVICE)
                    yb = yb.to(DEVICE)
                    preds = model(xb).cpu().numpy().ravel()
                    val_preds.append(preds)
                    val_targets.append(yb.cpu().numpy().ravel())

            val_preds = np.concatenate(val_preds)
            val_targets = np.concatenate(val_targets)

            # mse en escala (para pruning); convertimos a reales para devolver al final
            mse_scaled = mean_squared_error(val_targets, val_preds)

            # reportar a Optuna (para pruner)
            trial.report(mse_scaled, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            # early stopping en la métrica real (convertir pred a real)
            val_preds_real = scaler_y.inverse_transform(val_preds.reshape(-1, 1)).ravel()
            val_targets_real = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).ravel()
            mse_real = mean_squared_error(val_targets_real, val_preds_real)

            # actualizar early stopping
            if mse_real < best_val - 1e-6:
                best_val = mse_real
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= PATIENCE:
                break

        # Al terminar el entrenamiento del trial, evaluar en el set de validación completo (usamos X_val_orig)
        model.eval()
        with torch.no_grad():
            preds_val_scaled = model(X_val_t).cpu().numpy().ravel()
        preds_val_real = scaler_y.inverse_transform(preds_val_scaled.reshape(-1, 1)).ravel()
        mse_val_real = mean_squared_error(y_val_orig.ravel(), preds_val_real)

        return mse_val_real

    return objective

# -------------------------
# Loop por dataset: crear estudio y optimizar
# -------------------------
best_hyperparams_cnn = {"cnn": {}}

for ds_name, data in cnn_data_dict.items():
    print("\n" + "="*40)
    print("Dataset:", ds_name)
    print("="*40)

    X_train = data['train']['X']
    y_train = data['train']['Y'].ravel()
    X_test  = data['test']['X']
    y_test  = data['test']['Y'].ravel()

    # creamos un pequeño validation set a partir del test (o podrías usar test directamente como val)
    # aquí usamos X_test,y_test como conjunto de validación final
    objective = make_objective(X_train, y_train, X_test, y_test)

    # Crear estudio con pruner
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2))

    print(f"Iniciando optimización (trials={N_TRIALS_PER_DATASET}) para dataset {ds_name} ...")
    study.optimize(objective, n_trials=N_TRIALS_PER_DATASET, n_jobs=1, show_progress_bar=True)

    print("Best params:", study.best_params)
    print("Best MSE (real, on X_test):", study.best_value)

    best_hyperparams_cnn["cnn"][ds_name] = {
        "best_params": study.best_params,
        "best_value": study.best_value
    }

    # liberar VRAM y cache posible entre datasets
    torch.cuda.empty_cache()

C:\Users\mjime\anaconda3\envs\tesiskarlo-tf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-12-14 14:10:27,505] A new study created in memory with name: no-name-cdcff295-daff-4d75-b25e-bbc889158948


Device: cuda

Dataset: orig
Iniciando optimización (trials=25) para dataset orig ...


  0%|                                                                                           | 0/25 [00:00<?, ?it/s]


CPU Núcleos: [0.5, 0.7, 9.3, 0.4, 5.1, 0.8, 2.7, 1.1, 10.1, 0.0, 6.8, 0.1, 7.9, 0.2, 3.5, 1.2, 0.2, 0.3, 0.2, 0.0, 0.0, 0.1, 0.7, 0.0, 0.5, 0.2, 0.0, 0.0, 0.4, 0.2, 0.2, 0.1]
RAM Uso: 40.3%
GPU Uso: 0.0% | Mem: 7.5% | Temp: 50.0°C

CPU Núcleos: [10.6, 2.0, 3.3, 9.5, 11.5, 5.7, 13.8, 4.2, 11.1, 0.5, 4.6, 10.6, 21.1, 1.2, 43.5, 1.1, 4.0, 1.6, 0.5, 0.2, 0.2, 0.2, 0.4, 3.7, 2.0, 0.9, 0.9, 0.5, 2.1, 2.1, 0.9, 2.9]
RAM Uso: 38.2%
GPU Uso: 45.0% | Mem: 7.4% | Temp: 55.0°C

CPU Núcleos: [5.6, 1.6, 3.1, 4.4, 5.7, 5.6, 38.1, 1.6, 4.1, 1.3, 1.9, 3.6, 7.9, 0.6, 22.2, 0.6, 3.9, 0.9, 3.2, 0.1, 0.8, 0.0, 0.4, 3.3, 2.6, 1.1, 0.4, 0.4, 10.3, 3.7, 1.2, 0.6]
RAM Uso: 39.7%
GPU Uso: 33.0% | Mem: 7.3% | Temp: 59.0°C

CPU Núcleos: [12.8, 1.2, 7.7, 1.2, 9.4, 1.9, 8.4, 5.3, 2.6, 1.2, 13.9, 1.4, 20.9, 0.9, 22.8, 1.2, 3.5, 1.7, 0.7, 0.2, 3.3, 1.5, 0.5, 1.1, 4.1, 3.3, 0.6, 0.7, 11.2, 1.6, 2.7, 4.0]
RAM Uso: 40.0%
GPU Uso: 0.0% | Mem: 7.2% | Temp: 58.0°C

CPU Núcleos: [9.6, 2.0, 10.2, 3.5, 4.2, 19.0, 17.1, 2.2, 

Best trial: 0. Best value: 40.4951:   4%|█▉                                             | 1/25 [01:31<36:46, 91.93s/it]

...

[I 2025-12-14 14:24:00,969] Trial 11 pruned. 

CPU Núcleos: [8.3, 2.4, 14.7, 2.8, 6.9, 9.1, 8.0, 15.9, 14.2, 4.3, 4.8, 7.2, 10.5, 6.1, 25.9, 0.7, 0.5, 2.2, 2.7, 0.8, 1.7, 1.2, 0.5, 0.7, 1.5, 1.6, 1.5, 0.8, 3.6, 5.6, 2.3, 2.3]
RAM Uso: 36.8%
GPU Uso: 10.0% | Mem: 10.4% | Temp: 56.0°C

CPU Núcleos: [8.6, 1.4, 16.9, 1.9, 9.9, 2.1, 12.4, 3.7, 5.5, 1.1, 9.0, 1.3, 20.0, 2.0, 46.5, 1.6, 4.1, 0.3, 3.8, 2.0, 1.5, 1.0, 1.2, 0.5, 2.7, 1.2, 0.9, 0.8, 2.6, 1.6, 1.6, 2.3]
RAM Uso: 40.5%
GPU Uso: 23.0% | Mem: 10.6% | Temp: 56.0°C

CPU Núcleos: [8.6, 2.7, 9.4, 1.9, 18.3, 5.1, 4.4, 10.8, 8.9, 0.5, 2.3, 8.1, 15.5, 5.4, 18.4, 1.2, 2.5, 0.5, 0.1, 2.9, 1.8, 2.3, 0.5, 0.3, 1.6, 1.6, 1.2, 0.4, 1.9, 1.2, 1.2, 1.8]
RAM Uso: 36.1%
GPU Uso: 0.0% | Mem: 10.7% | Temp: 57.0°C


Best trial: 4. Best value: 19.5012:  48%|██████████████████████                        | 12/25 [14:43<15:57, 73.65s/it]

[W 2025-12-14 14:25:11,307] Trial 12 failed with parameters: {'n_filters': 64, 'kernel_size': 5, 'n_conv_blocks': 1, 'hidden': 192, 'dropout': 0.038944062580358096, 'lr': 0.00016337770859004382, 'weight_decay': 6.717715238348037e-05, 'batch_size': 32} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\mjime\anaconda3\envs\tesiskarlo-tf\lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\mjime\AppData\Local\Temp\ipykernel_3052\2334008065.py", line 152, in objective
    for xb, yb in val_loader:
  File "C:\Users\mjime\anaconda3\envs\tesiskarlo-tf\lib\site-packages\torch\utils\data\dataloader.py", line 733, in __next__
    data = self._next_data()
  File "C:\Users\mjime\anaconda3\envs\tesiskarlo-tf\lib\site-packages\torch\utils\data\dataloader.py", line 1491, in _next_data
    idx, data = self._get_data()
  File "C:\Users\mjime\anaconda3\envs\tesiskarlo-tf\lib\site-pa



KeyboardInterrupt




CPU Núcleos: [20.7, 1.6, 8.1, 2.4, 26.8, 2.7, 4.5, 6.9, 5.1, 1.2, 6.9, 12.4, 6.4, 1.9, 10.7, 1.3, 2.0, 0.9, 0.4, 0.9, 1.2, 1.2, 0.9, 0.1, 1.5, 1.6, 0.4, 0.5, 1.6, 1.7, 1.5, 2.2]
RAM Uso: 41.1%
GPU Uso: 12.0% | Mem: 10.6% | Temp: 55.0°C


In [11]:
# -------------------------
# Guardar resultados
# -------------------------
# si ya tienes best_hyperparams (por ejemplo RF + SVR), carga y fusiona, sino guarda sólo cnn:
outfile = "best_hyperparams_all_models.pkl"

# si existe archivo previo, lo cargamos y actualizamos
if os.path.exists(outfile):
    with open(outfile, "rb") as f:
        all_hp = pickle.load(f)
else:
    all_hp = {}

all_hp.update(best_hyperparams_cnn)  # agrega/actualiza clave "cnn"

with open(outfile, "wb") as f:
    pickle.dump(all_hp, f)

print("Guardado en:", outfile)
print("Resumen final (keys):", list(all_hp.keys()))

# Detener monitor
monitor_flag = False
monitor_thread.join()

Guardado en: best_hyperparams_all_models.pkl
Resumen final (keys): ['random_forest', 'svr', 'cnn']
